# BGP Feature Correlation Analysis

This notebook analyzes correlations between the 25 extracted BGP features to identify:
1. Highly correlated feature pairs (redundant features)
2. Feature groups/clusters based on correlation
3. Which features are related to which features

**Input**: CSV file from `feature_exctration.ipynb`  
**Output**: Correlation matrices, heatmaps, and feature relationship summaries saved to `../output/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION - Update these paths to match your environment
# ============================================================
INPUT_FILE = "/home/smotaali/BGP_Traffic_Generation/results/20251208_223239_exctracted_1s.csv"
OUTPUT_DIR = "../output"

# Correlation thresholds
HIGH_CORR_THRESHOLD = 0.7   # Features with |correlation| >= this are "highly correlated"
MODERATE_CORR_THRESHOLD = 0.4  # Features with |correlation| >= this are "moderately correlated"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Input: {INPUT_FILE}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Load the extracted features
df = pd.read_csv(INPUT_FILE)

# The 25 BGP features from feature_exctration.ipynb
FEATURE_COLS = [
    'announcements', 'withdrawals', 'nlri_ann', 'dups',
    'origin_0', 'origin_2', 'origin_changes',
    'as_path_max', 'unique_as_path_max',
    'edit_distance_avg', 'edit_distance_max',
    'edit_distance_dict_0', 'edit_distance_dict_1', 'edit_distance_dict_2',
    'edit_distance_dict_4', 'edit_distance_dict_6',
    'edit_distance_unique_dict_0', 'edit_distance_unique_dict_1',
    'imp_wd', 'imp_wd_spath', 'imp_wd_dpath',
    'number_rare_ases', 'rare_ases_avg',
    'flaps', 'nadas'
]

# Keep only features that exist in the dataframe
available_features = [f for f in FEATURE_COLS if f in df.columns]
missing_features = [f for f in FEATURE_COLS if f not in df.columns]

if missing_features:
    print(f"Warning - missing features: {missing_features}")

df_features = df[available_features].copy()

print(f"Loaded {len(df)} rows, {len(available_features)} features")
print(f"\nFeature summary:")
df_features.describe().round(3)

## 1. Pearson Correlation Matrix

In [ ]:
# Compute Pearson correlation
corr_matrix = df_features.corr(method='pearson')

# Save correlation matrix
corr_matrix.to_csv(os.path.join(OUTPUT_DIR, 'pearson_correlation_matrix.csv'))

# Plot full heatmap
fig, ax = plt.subplots(figsize=(18, 15))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # Upper triangle mask
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title('Pearson Correlation Matrix - BGP Features', fontsize=16, pad=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'pearson_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: pearson_correlation_heatmap.png")

## 2. Spearman Rank Correlation (captures non-linear relationships)

In [ ]:
# Compute Spearman correlation
spearman_corr = df_features.corr(method='spearman')

# Save
spearman_corr.to_csv(os.path.join(OUTPUT_DIR, 'spearman_correlation_matrix.csv'))

fig, ax = plt.subplots(figsize=(18, 15))
mask = np.triu(np.ones_like(spearman_corr, dtype=bool), k=1)
sns.heatmap(spearman_corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title('Spearman Rank Correlation Matrix - BGP Features', fontsize=16, pad=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'spearman_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: spearman_correlation_heatmap.png")

## 3. Highly Correlated Feature Pairs

Identify which features are strongly related (potentially redundant).

In [ ]:
def get_correlated_pairs(corr_mat, threshold):
    """Extract feature pairs above a correlation threshold."""
    pairs = []
    cols = corr_mat.columns
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            val = corr_mat.iloc[i, j]
            if abs(val) >= threshold:
                pairs.append({
                    'Feature_1': cols[i],
                    'Feature_2': cols[j],
                    'Pearson': corr_matrix.loc[cols[i], cols[j]],
                    'Spearman': spearman_corr.loc[cols[i], cols[j]],
                    'Abs_Pearson': abs(corr_matrix.loc[cols[i], cols[j]])
                })
    return pd.DataFrame(pairs).sort_values('Abs_Pearson', ascending=False)

# Highly correlated pairs
high_corr_pairs = get_correlated_pairs(corr_matrix, HIGH_CORR_THRESHOLD)
moderate_corr_pairs = get_correlated_pairs(corr_matrix, MODERATE_CORR_THRESHOLD)

print(f"=== HIGHLY CORRELATED PAIRS (|r| >= {HIGH_CORR_THRESHOLD}) ===")
print(f"Found {len(high_corr_pairs)} pairs\n")
if not high_corr_pairs.empty:
    display(high_corr_pairs[['Feature_1', 'Feature_2', 'Pearson', 'Spearman']].round(3))
else:
    print("No highly correlated pairs found.")

print(f"\n=== MODERATELY CORRELATED PAIRS (|r| >= {MODERATE_CORR_THRESHOLD}) ===")
print(f"Found {len(moderate_corr_pairs)} pairs\n")
if not moderate_corr_pairs.empty:
    display(moderate_corr_pairs[['Feature_1', 'Feature_2', 'Pearson', 'Spearman']].round(3))

# Save to CSV
high_corr_pairs.to_csv(os.path.join(OUTPUT_DIR, 'highly_correlated_pairs.csv'), index=False)
moderate_corr_pairs.to_csv(os.path.join(OUTPUT_DIR, 'moderately_correlated_pairs.csv'), index=False)
print("\nSaved: highly_correlated_pairs.csv, moderately_correlated_pairs.csv")

## 4. Hierarchical Clustering of Features

Group features into clusters based on correlation similarity.  
Features in the same cluster are closely related to each other.

In [ ]:
# Convert correlation to distance (1 - |correlation|)
corr_abs = corr_matrix.abs()
distance_matrix = 1 - corr_abs

# Ensure diagonal is 0 and matrix is symmetric
np.fill_diagonal(distance_matrix.values, 0)
distance_matrix = (distance_matrix + distance_matrix.T) / 2

# Clip any small negative values from floating point errors
distance_matrix = distance_matrix.clip(lower=0)

# Hierarchical clustering
condensed_dist = squareform(distance_matrix.values)
linkage_matrix = linkage(condensed_dist, method='average')

# Plot dendrogram
fig, ax = plt.subplots(figsize=(16, 8))
dendrogram(linkage_matrix, labels=available_features, leaf_rotation=90, leaf_font_size=10, ax=ax)
ax.set_title('Feature Clustering Dendrogram (based on correlation distance)', fontsize=14)
ax.set_ylabel('Distance (1 - |correlation|)', fontsize=12)
ax.axhline(y=1 - HIGH_CORR_THRESHOLD, color='r', linestyle='--', label=f'High corr threshold ({HIGH_CORR_THRESHOLD})')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_clustering_dendrogram.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: feature_clustering_dendrogram.png")

In [ ]:
# Cut the dendrogram to form clusters
cluster_threshold = 1 - MODERATE_CORR_THRESHOLD  # distance threshold
clusters = fcluster(linkage_matrix, t=cluster_threshold, criterion='distance')

# Build cluster membership
feature_clusters = pd.DataFrame({
    'Feature': available_features,
    'Cluster': clusters
}).sort_values('Cluster')

print("=== FEATURE GROUPS (clusters of related features) ===")
print(f"Cut at distance = {cluster_threshold:.2f} (correlation >= {MODERATE_CORR_THRESHOLD})\n")

for cluster_id in sorted(feature_clusters['Cluster'].unique()):
    members = feature_clusters[feature_clusters['Cluster'] == cluster_id]['Feature'].tolist()
    print(f"Group {cluster_id} ({len(members)} features):")
    for m in members:
        print(f"  - {m}")
    print()

feature_clusters.to_csv(os.path.join(OUTPUT_DIR, 'feature_cluster_assignments.csv'), index=False)
print("Saved: feature_cluster_assignments.csv")

## 5. Feature Relationship Network

Visualize which features are connected (correlated) as a network graph.

In [ ]:
import networkx as nx

# Build a graph of feature relationships
G = nx.Graph()
G.add_nodes_from(available_features)

# Add edges for correlated pairs
for i in range(len(available_features)):
    for j in range(i + 1, len(available_features)):
        r = corr_matrix.iloc[i, j]
        if abs(r) >= MODERATE_CORR_THRESHOLD:
            G.add_edge(available_features[i], available_features[j],
                       weight=abs(r), correlation=r)

# Color nodes by cluster
cluster_map = dict(zip(feature_clusters['Feature'], feature_clusters['Cluster']))
cluster_colors = plt.cm.Set3(np.linspace(0, 1, len(set(clusters))))
node_colors = [cluster_colors[cluster_map.get(n, 1) - 1] for n in G.nodes()]

fig, ax = plt.subplots(figsize=(16, 12))
pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

# Draw edges with width proportional to correlation strength
edges = G.edges(data=True)
if edges:
    edge_weights = [d['weight'] * 3 for _, _, d in edges]
    edge_colors = ['red' if d['correlation'] < 0 else 'blue' for _, _, d in edges]
    edge_alphas = [d['weight'] * 0.8 for _, _, d in edges]
    nx.draw_networkx_edges(G, pos, width=edge_weights, edge_color=edge_colors,
                           alpha=0.5, ax=ax)

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=800,
                        edgecolors='black', linewidths=1, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)

# Edge labels for high correlations
high_edge_labels = {(u, v): f"{d['correlation']:.2f}"
                    for u, v, d in edges if d['weight'] >= HIGH_CORR_THRESHOLD}
nx.draw_networkx_edge_labels(G, pos, edge_labels=high_edge_labels, font_size=7, ax=ax)

ax.set_title(f'Feature Relationship Network (edges: |r| >= {MODERATE_CORR_THRESHOLD})\n'
             f'Blue = positive correlation, Red = negative correlation', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_relationship_network.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: feature_relationship_network.png")

## 6. Feature-to-Feature Relationship Summary Table

For each feature, list which other features it is related to.

In [ ]:
# Build a summary: for each feature, list its correlated partners
summary_rows = []

for feat in available_features:
    correlations = corr_matrix[feat].drop(feat).abs().sort_values(ascending=False)

    high = correlations[correlations >= HIGH_CORR_THRESHOLD]
    moderate = correlations[(correlations >= MODERATE_CORR_THRESHOLD) & (correlations < HIGH_CORR_THRESHOLD)]

    high_list = ', '.join([f"{f} ({corr_matrix[feat][f]:.2f})" for f in high.index])
    moderate_list = ', '.join([f"{f} ({corr_matrix[feat][f]:.2f})" for f in moderate.index])

    summary_rows.append({
        'Feature': feat,
        'Cluster': cluster_map.get(feat, '-'),
        'Highly_Correlated (|r|>=0.7)': high_list if high_list else 'None',
        'Moderately_Correlated (0.4<=|r|<0.7)': moderate_list if moderate_list else 'None',
        'Max_Abs_Correlation': correlations.max(),
        'Num_High_Corr': len(high),
        'Num_Moderate_Corr': len(moderate)
    })

summary_df = pd.DataFrame(summary_rows).sort_values('Max_Abs_Correlation', ascending=False)

print("=== FEATURE RELATIONSHIP SUMMARY ===")
print(f"(High: |r| >= {HIGH_CORR_THRESHOLD}, Moderate: |r| >= {MODERATE_CORR_THRESHOLD})\n")

for _, row in summary_df.iterrows():
    print(f"--- {row['Feature']} (Cluster {row['Cluster']}) ---")
    print(f"  Highly correlated with: {row['Highly_Correlated (|r|>=0.7)']}")
    print(f"  Moderately correlated with: {row['Moderately_Correlated (0.4<=|r|<0.7)']}")
    print()

summary_df.to_csv(os.path.join(OUTPUT_DIR, 'feature_relationship_summary.csv'), index=False)
print("Saved: feature_relationship_summary.csv")

## 7. Clustered Heatmap (reordered by similarity)

In [ ]:
# Clustered heatmap - reorders features by similarity automatically
g = sns.clustermap(corr_matrix, method='average', cmap='RdBu_r',
                   center=0, vmin=-1, vmax=1, figsize=(16, 14),
                   annot=True, fmt='.2f', annot_kws={'size': 6},
                   linewidths=0.3)
g.fig.suptitle('Clustered Correlation Heatmap (features reordered by similarity)', 
               fontsize=14, y=1.02)
plt.savefig(os.path.join(OUTPUT_DIR, 'clustered_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: clustered_correlation_heatmap.png")

## 8. Scatter Plots of Top Correlated Pairs

In [ ]:
# Plot scatter plots for top correlated pairs
if not high_corr_pairs.empty:
    top_pairs = high_corr_pairs.head(6)  # Top 6 pairs
    n_pairs = len(top_pairs)
    n_cols = min(3, n_pairs)
    n_rows = (n_pairs + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
    if n_pairs == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for idx, (_, row) in enumerate(top_pairs.iterrows()):
        ax = axes[idx]
        f1, f2 = row['Feature_1'], row['Feature_2']
        ax.scatter(df_features[f1], df_features[f2], alpha=0.3, s=10)
        ax.set_xlabel(f1, fontsize=10)
        ax.set_ylabel(f2, fontsize=10)
        ax.set_title(f'r = {row["Pearson"]:.3f}', fontsize=11)

    # Hide unused subplots
    for idx in range(n_pairs, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle('Scatter Plots of Top Correlated Feature Pairs', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'top_correlated_scatter_plots.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: top_correlated_scatter_plots.png")
else:
    print("No highly correlated pairs to plot.")

## 9. Final Report: Feature Redundancy Recommendations

In [ ]:
print("="*70)
print("FEATURE CORRELATION ANALYSIS - FINAL REPORT")
print("="*70)
print(f"\nTotal features analyzed: {len(available_features)}")
print(f"Total samples (time windows): {len(df)}")
print(f"\nHighly correlated pairs (|r| >= {HIGH_CORR_THRESHOLD}): {len(high_corr_pairs)}")
print(f"Moderately correlated pairs (|r| >= {MODERATE_CORR_THRESHOLD}): {len(moderate_corr_pairs)}")
print(f"Feature clusters identified: {len(set(clusters))}")

# Identify potentially redundant features
if not high_corr_pairs.empty:
    print(f"\n--- Potentially Redundant Features (|r| >= {HIGH_CORR_THRESHOLD}) ---")
    print("Consider removing one feature from each pair below:")
    for _, row in high_corr_pairs.iterrows():
        direction = 'positive' if row['Pearson'] > 0 else 'negative'
        print(f"  {row['Feature_1']}  <-->  {row['Feature_2']}  "
              f"(Pearson: {row['Pearson']:.3f}, {direction})")

# Independent features (low correlation with everything)
independent_features = []
for feat in available_features:
    max_corr = corr_matrix[feat].drop(feat).abs().max()
    if max_corr < MODERATE_CORR_THRESHOLD:
        independent_features.append((feat, max_corr))

if independent_features:
    print(f"\n--- Independent Features (max |r| < {MODERATE_CORR_THRESHOLD}) ---")
    print("These features capture unique information:")
    for feat, max_corr in independent_features:
        print(f"  {feat} (max |r| = {max_corr:.3f})")

print(f"\n--- Output files saved to {OUTPUT_DIR}/ ---")
for f in os.listdir(OUTPUT_DIR):
    print(f"  {f}")
print("\nDone!")